# 02 — Data Cleaning

## UAE Used Car Market Analysis

Starting point: `data/processed/uae_cars_combined_deduped.csv` — 28,109
listings across CarSwitch, AutoTraders.ae, and OpenSooq, already
deduplicated (see `01_scraping.ipynb`).

The reusable pieces — brand normalization, the brand-segment
classification, city standardization — live in `src/finalize_dataset.py`
so they can be reproduced outside a notebook too. This notebook imports
them and walks through each decision, showing the actual before/after
at each step rather than just calling one function and trusting it.

In [17]:
import sys
import pandas as pd
import numpy as np

sys.path.append("../scraper")
import finalize_dataset as fd

pd.set_option("display.max_columns", None)

data_dir = "../data"
df = pd.read_csv(f"{data_dir}/processed/uae_cars_combined_deduped.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (28109, 15)


,name,brand,brand_segment,model,year,car_age,mileage_km,mileage_per_year,price_aed,price_tier,city,url,source,listing_id,transmission
0,2020 Dodge RAM 1500 Limited,Dodge,Mid-range,RAM,2020,6.0,198250.0,33041.666667,79500.0,Mid-range,Abu Dhabi,https://carswitch.com/abudhabi/used-car/dodge/...,carswitch,NaN,NaN
1,2022 MINI Cooper Twin Power Turbo,MINI,Premium,Cooper,2022,4.0,50350.0,12587.500000,56000.0,Mid-range,Dubai,https://carswitch.com/dubai/used-car/mini/coop...,carswitch,NaN,NaN
2,2017 Infiniti QX70,Infiniti,Premium,QX70,2017,9.0,76000.0,8444.444444,35000.0,Mid-range,Abu Dhabi,https://carswitch.com/abudhabi/used-car/infini...,carswitch,NaN,NaN
3,2018 Jeep Grand Cherokee Limited,Jeep,Mid-range,Grand Cherokee,2018,8.0,110600.0,13825.000000,40000.0,Mid-range,Dubai,https://carswitch.com/dubai/used-car/jeep/gran...,carswitch,NaN,NaN
4,2024 Tesla Model Y Long Range Dual Motor AWD,Tesla,Mid-range,Model Y,2024,2.0,69500.0,34750.000000,120900.0,Premium,Abu Dhabi,https://carswitch.com/abudhabi/used-car/tesla/...,carswitch,NaN,NaN


In [18]:
df.isna().sum()

name                    0
brand                   0
brand_segment       25789
model                   0
year                    0
car_age             25789
mileage_km           3930
mileage_per_year    25789
price_aed            3835
price_tier          25789
city                    0
url                     0
source                  0
listing_id           2320
transmission        28109
dtype: int64

A few things stand out right away. `car_age`, `price_tier`,
`mileage_per_year`, and `brand_segment` are only filled in for 2,320 rows
— that's exactly the CarSwitch count, since that file came in pre-cleaned
from an earlier step and the other two sources haven't been through any
feature engineering yet. `transmission` needs a look too, and
`listing_id` is missing for that same 2,320-row block.

In [19]:
print("transmission non-null count:", df["transmission"].notna().sum(), "/", len(df))
print("listing_id null count:", df["listing_id"].isna().sum())

transmission non-null count: 0 / 28109
listing_id null count: 2320


## Dropping `transmission`

Zero non-null values across all 28,109 rows. Not one usable data point —
dropping it.

In [20]:
df = df.drop(columns=["transmission", "car_age", "price_tier", "mileage_per_year", "brand_segment"])
df.shape

(28109, 10)

## Backfilling `listing_id`

Regenerating an ID for the CarSwitch rows the same way the scrapers do it
for everything else — a short hash of source + URL — so every row has
something stable to key off of.

In [21]:
missing_id = df["listing_id"].isna()
df.loc[missing_id, "listing_id"] = df.loc[missing_id].apply(
    lambda r: fd.make_listing_id(r["source"], r["url"]), axis=1
)
print(f"Backfilled {missing_id.sum()} listing_id values")
df["listing_id"].isna().sum()

Backfilled 2320 listing_id values


np.int64(0)

## Brand name normalization

Three sites, three slightly different ways of writing the same brand.
Before any brand-level grouping means anything, these need to collapse
into one:

| What showed up | Canonical form |
|---|---|
| `Mercedes-Benz`, `Mercedes Benz`, `Mercedes` | `Mercedes-Benz` |
| `MINI`, `Mini` | `MINI` |
| `RAM`, `Ram` | `Ram` |
| `BAIC`, `Baic` | `BAIC` |
| `FIAT`, `Fiat` | `Fiat` |
| `Rox`, `ROX` | `ROX` |
| `SsangYong`, `Ssangyong` | `SsangYong` |
| `Zeekr`, `ZEEKR` | `Zeekr` |
| `Jaecoo`, `JAECOO` | `Jaecoo` |
| `Avatar`, `Avatr` | `Avatr` |
| plus a few more (full list in `finalize_dataset.BRAND_NORMALIZE`) |

In [22]:
print("Before:")
print(df["brand"].value_counts().loc[lambda s: s.index.str.lower().isin(
    ["mercedes-benz", "mercedes benz", "mercedes", "mini", "ram"]
)])

df["brand"] = df["brand"].apply(fd.normalize_brand_text)

print("\nAfter:")
print(df["brand"].value_counts().loc[lambda s: s.index.isin(["Mercedes-Benz", "MINI", "Ram"])])

Before:
brand
Mercedes-Benz    1723
Mercedes Benz     718
Mini              142
Mercedes          113
MINI               73
Ram                56
RAM                 3
Name: count, dtype: int64

After:
brand
Mercedes-Benz    2554
MINI              215
Ram                59
Name: count, dtype: int64


## "Range Rover" isn't a brand

It's a Land Rover model, but a chunk of AutoTraders.ae dealer listings
had written it straight into the brand field — probably how the dealer
typed it in when posting. Reassigning those rows to `brand: Land Rover`
and folding "Range Rover" into the model name where it isn't there
already.

In [23]:
print(f"Rows with brand == 'Range Rover' before fix: {(df['brand'] == 'Range Rover').sum()}")

df[["brand", "model"]] = df.apply(fd.fix_range_rover_brand, axis=1)

print(f"Rows with brand == 'Range Rover' after fix: {(df['brand'] == 'Range Rover').sum()}")
print(f"Land Rover total now: {(df['brand'] == 'Land Rover').sum()}")
df.loc[df["model"].str.contains("Range Rover", na=False), ["brand", "model"]].head()

Rows with brand == 'Range Rover' before fix: 41
Rows with brand == 'Range Rover' after fix: 0
Land Rover total now: 1054


,brand,model
75,Land Rover,Range Rover Evoque
96,Land Rover,Range Rover Velar
120,Land Rover,Range Rover Velar
140,Land Rover,Range Rover Evoque
155,Land Rover,Range Rover Evoque


## Model name casing

The same splitting-across-spellings problem shows up in model names too
— `KICKS` versus `Kicks` was the one that actually caught my eye later,
in the EDA notebook, where it showed up as two separate bars on a chart
that should have had one. Different sources just format model text
differently.

Hand-coding fixes for every acronym-style model name (`IS`, `GS`, `X5`,
`C-Class`, `CR-V`...) wasn't appealing — a blanket `.title()` would wreck
half of those. Instead, for each `(brand, lowercased model)` pair, this
picks whichever casing shows up most often and uses that everywhere.
Data-driven, so it doesn't need babysitting every time a new brand or
model gets scraped in.

In [24]:
print(f"Before: {(df['brand'] + ' ' + df['model']).nunique()} unique brand+model combos (case-sensitive)")

df = fd.normalize_model_casing(df)

print(f"After: {(df['brand'] + ' ' + df['model']).nunique()} unique brand+model combos (case-sensitive)")

# the case that actually surfaced this
print(df.loc[(df["brand"] == "Nissan") & (df["model"].str.lower() == "kicks"), "model"].value_counts())

Before: 2043 unique brand+model combos (case-sensitive)
After: 2022 unique brand+model combos (case-sensitive)
model
KICKS    187
Name: count, dtype: int64


## "Brand new" listings with no mileage

Some AutoTraders.ae listings are missing a mileage figure entirely — no
"KM" chip shown at all. Checking the actual titles, these are mostly
unregistered/new vehicles ("... BRAND NEW"), where zero km is the honest
answer rather than a gap in the data. Only imputing it where the title
says so explicitly — not extending that assumption to the rest of the
missing-mileage rows.

In [25]:
brand_new_mask = df["mileage_km"].isna() & df["name"].astype(str).str.contains("brand new", case=False, na=False)
print(f"Setting mileage_km = 0 for {brand_new_mask.sum()} confirmed 'brand new' listings")

df.loc[brand_new_mask, "mileage_km"] = 0

print(f"Remaining null mileage_km (genuinely unknown): {df['mileage_km'].isna().sum()}")

Setting mileage_km = 0 for 45 confirmed 'brand new' listings
Remaining null mileage_km (genuinely unknown): 3885


## Price and mileage outliers

Before computing anything derived from these fields, it's worth actually
looking at the extreme values rather than trusting a threshold blindly.

Forty-two rows had a price of essentially zero — AutoTraders.ae only,
every one of them otherwise a completely normal listing (a Lamborghini
Urus, a Nissan Rogue, a Range Rover) with a believable mileage. A lot of
them break to the exact same value, `1`, which reads like a
placeholder shown before the real price finishes loading on the page,
not a scraping error on this end.

Eleven rows had mileage in the millions of kilometers — same source,
different field. A handful of listing cards clearly use a different
internal layout than the standard one, and the mileage parser grabs the
wrong digits from that variant. Two of the affected listings had
Arabic-language titles, which points toward a specific older or
differently-formatted listing style rather than anything random.

Both issues sit around 0.2% of AutoTraders.ae's rows — small enough that
chasing a perfect fix for every template quirk isn't worth it. Flagging
and excluding these from price/mileage math, while keeping the rows
themselves (brand, model, year are all still fine) for anything based on
listing counts or composition.

In [26]:
MIN_PLAUSIBLE_PRICE = 1_000
MAX_PLAUSIBLE_MILEAGE = 1_000_000

df["price_suspect"] = df["price_aed"].notna() & (df["price_aed"] < MIN_PLAUSIBLE_PRICE)
df["mileage_suspect"] = df["mileage_km"].notna() & (df["mileage_km"] > MAX_PLAUSIBLE_MILEAGE)

print(f"price_suspect: {df['price_suspect'].sum()} rows")
print(f"mileage_suspect: {df['mileage_suspect'].sum()} rows")
print(f"\nUsable for price analysis: {(df['price_aed'].notna() & ~df['price_suspect']).sum()} / {len(df)} "
      f"({100*(df['price_aed'].notna() & ~df['price_suspect']).mean():.1f}%)")

price_suspect: 42 rows
mileage_suspect: 11 rows

Usable for price analysis: 24232 / 28109 (86.2%)


## Feature engineering

- **`car_age`** — 2026 minus the model year, floored at 0
- **`price_tier`** — Budget (under 30K AED), Mid-range (30K-100K),
  Premium (100K-300K), Luxury (over 300K), computed with the suspect
  prices excluded so a broken "AED 1" listing doesn't get bucketed as
  Budget
- **`mileage_per_year`** — mileage divided by car age, with brand-new
  cars (age 0) using a denominator of 1 so it doesn't divide by zero

In [27]:
CURRENT_YEAR = 2026
df["car_age"] = (CURRENT_YEAR - df["year"]).clip(lower=0)

price_for_tier = df["price_aed"].where(~df["price_suspect"])
df["price_tier"] = pd.cut(
    price_for_tier,
    bins=[0, 30_000, 100_000, 300_000, float("inf")],
    labels=["Budget", "Mid-range", "Premium", "Luxury"],
    right=False,
)

mileage_for_calc = df["mileage_km"].where(~df["mileage_suspect"])
denom = df["car_age"].replace(0, 1)
df["mileage_per_year"] = mileage_for_calc / denom

df[["year", "car_age", "price_aed", "price_tier", "mileage_km", "mileage_per_year"]].head()

,year,car_age,price_aed,price_tier,mileage_km,mileage_per_year
0,2020,6,79500.0,Mid-range,198250.0,33041.666667
1,2022,4,56000.0,Mid-range,50350.0,12587.500000
2,2017,9,35000.0,Mid-range,76000.0,8444.444444
3,2018,8,40000.0,Mid-range,110600.0,13825.000000
4,2024,2,120900.0,Premium,69500.0,34750.000000


## Brand segments

187 unique brand strings after normalization, everything from Toyota to
Bugatti to a handful of Chinese EV startups I'd never heard of before
this project. Four tiers:

- **Mass-market** — Toyota, Nissan, Hyundai, plus the budget Chinese and
  other economy brands
- **Mid-range** — American mainstream (Jeep, GMC, Dodge, Ram), a few
  older European marques, some entry-level EVs
- **Premium** — BMW, Audi, Lexus, Infiniti, premium EVs, entry-luxury
  brands generally
- **Luxury** — Ferrari, Bugatti, Rolls-Royce, and similar, plus a few
  boutique/bespoke manufacturers

Full mapping in `finalize_dataset.BRAND_SEGMENT_MAP`. A handful of
brands with only 1-3 listings each got a best guess rather than deep
research, since they carry basically no weight in any aggregate number —
easy enough to fix by hand if one looks wrong.

In [28]:
df["brand_segment"] = df["brand"].map(fd.BRAND_SEGMENT_MAP)

unmapped = df.loc[df["brand_segment"].isna() & df["brand"].notna(), "brand"].unique()
print(f"Unmapped brands: {len(unmapped)}")
assert len(unmapped) == 0, f"Found unmapped brands: {unmapped}"

df["brand_segment"].value_counts()

Unmapped brands: 0


brand_segment
Mass-market    16507
Premium         8660
Mid-range       1975
Luxury           967
Name: count, dtype: int64

In [29]:
df.groupby("source")["brand_segment"].value_counts(normalize=True).unstack().round(3)

brand_segment,Luxury,Mass-market,Mid-range,Premium
source,,,,
autotraders,0.048,0.553,0.071,0.327
carswitch,0.018,0.611,0.067,0.304
opensooq,0.012,0.646,0.069,0.273


The segment mix clearly isn't the same across the three sources —
that's worth carrying into the EDA notebook properly rather than treating
as a side note.

## City names

```
dubai -> Dubai
abudhabi / abu-dhabi / abu dhabi -> Abu Dhabi
sharjah -> Sharjah
ajman -> Ajman
ras-al-khaimah / rak -> Ras Al Khaimah
fujairah -> Fujairah
al-ain -> Al Ain
umm-al-quwain / um al quwain -> Umm Al Quwain
```

That last one (`um al quwain`, missing the second "m") is OpenSooq's own
spelling, not a typo introduced anywhere in this pipeline.

In [30]:
print("Before:")
print(df["city"].value_counts())

df["city"] = df["city"].apply(
    lambda x: fd.CITY_DISPLAY_MAP.get(str(x).strip().lower(), x) if pd.notna(x) else x
)

print("\nAfter:")
print(df["city"].value_counts())

Before:
city
Dubai             13142
Sharjah            5117
Abu Dhabi          4552
Ajman              2756
Al Ain             1471
Ras Al Khaimah      536
Fujairah            231
Um Al Quwain        212
Umm Al Quwain        92
Name: count, dtype: int64

After:
city
Dubai             13142
Sharjah            5117
Abu Dhabi          4552
Ajman              2756
Al Ain             1471
Ras Al Khaimah      536
Umm Al Quwain       304
Fujairah            231
Name: count, dtype: int64


## Saving it

In [31]:
print(f"Final shape: {df.shape}")
print()
print(df.isna().sum())
print()
print("By source:")
print(df["source"].value_counts())

Final shape: (28109, 16)

name                   0
brand                  0
model                  0
year                   0
mileage_km          3885
price_aed           3835
city                   0
url                    0
source                 0
listing_id             0
price_suspect          0
mileage_suspect        0
car_age                0
price_tier          3877
mileage_per_year    3896
brand_segment          0
dtype: int64

By source:
source
autotraders    16995
opensooq        8794
carswitch       2320
Name: count, dtype: int64


In [32]:
df.to_csv(f"{data_dir}/processed/uae_cars_final_clean.csv", index=False)
print(f"Saved {len(df)} rows -> {data_dir}/processed/uae_cars_final_clean.csv")

Saved 28109 rows -> ../data/processed/uae_cars_final_clean.csv


## Where things stand

| Step | Result |
|---|---|
| Dropped `transmission` | 100% empty, nothing to lose |
| Backfilled `listing_id` | 2,320 rows |
| Brand normalization | ~10 spelling/casing duplicates merged |
| Range Rover fix | 41 rows moved from brand to model |
| Model casing fix | Case-duplicate models merged |
| Mileage imputation | 45 confirmed "brand new" listings set to 0 km |
| Outlier flagging | 42 price rows + 11 mileage rows flagged, kept not dropped |
| Brand segmentation | 187 brands classified, none left unmapped |
| City standardization | 2 spelling variants merged |

28,109 rows, saved to `data/processed/uae_cars_final_clean.csv`.

Next: `03_eda.ipynb`.